In [12]:
import os

#直接使用GPT2模型来计算奖励,输入是一个文本序列，输出是该序列的奖励值。gpt+线性层
import torch
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer



class RewardHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        # llm最后输出的隐藏层的维度
        self.hidden_size = config.hidden_size
        # 线性层用来对llm最后输出的隐藏层给奖励
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        # 使用正态分布初始化权重
        nn.init.normal_(
            self.reward.weight,
            std=(1.0 / np.sqrt(self.hidden_size + 1))
        )
        # 将偏置初始化为0
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        # 给出奖励
        return self.reward(hidden_states)

class GPT2RewardHead(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        self.reward_head = RewardHead(self.llm.config)
        # # 冻结 GPT-2 的所有参数
        # for param in self.llm.parameters():
        #     param.requires_grad = False
        # # 将 GPT-2 设置为评估模式（影响 dropout 和 batchnorm）
        # self.llm.eval()

    def forward(self, input_ids, attention_mask):
        transformer_outputs = self.llm.forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        last_hidden_state = transformer_outputs.hidden_states[-1] # 获取最后一层的隐藏状态，而不是输出的logits
        # 给出奖励
        reward = self.reward_head(last_hidden_state).squeeze(-1)
        # sigmoid用来将奖励搞到(-1,1)范围内
        return torch.sigmoid(reward)

model_path = "C:\\Users\lihaodong\.cache\modelscope\hub\models\Fengshenbang\Wenzhong-GPT2-110M-chinese-v2"
model = GPT2RewardHead(model_path)


In [13]:
from functools import partial
#加载tokenizer
from datasets import load_dataset
from transformers import AutoTokenizer


# 定义分词函数：接收一个 batch 字典，返回一个字典（键是新的列名，值是列表）
def tokenize(batch, tokenizer, REWARD_TOKEN_ID):
    # 分词
    contexts = tokenizer(batch['context'])
    batch_size = len(contexts['input_ids'])  # 批次大小

    # 初始化得分和得分位置（长度 = batch_size）
    contexts["score"] = [0.0] * batch_size
    contexts["score_index"] = [0] * batch_size

    for i in range(batch_size):
        # 在每条文本末尾添加 reward token（eos token）
        contexts['input_ids'][i].append(REWARD_TOKEN_ID)
        contexts['attention_mask'][i].append(1)

        # 评分：label 是 1（正向）或 0（负向）
        contexts['score'][i] = float(batch['label'][i])

        # 记录 reward token 的索引（最后一个位置）
        contexts['score_index'][i] = len(contexts['input_ids'][i]) - 1

    return contexts


#清理数据长度国小的
def filter_short_samples(batch, min_length=10):
    # 计算输入 ID 的长度
    input_ids_length = len(batch['input_ids'])
    # 只有当长度大于或等于 min_length和小于等于100 时才保留样本
    return (input_ids_length >= min_length and input_ids_length <= 150)

def load_data(data_path,tokenizer):
    dataset = load_dataset('json', data_files={
        'train': data_path[0],
        'valid': data_path[1],
        'test': data_path[2]
    })
    # 查看结构
    print(dataset['train'][0])
    # 输出：{'idx': 0, 'context': '如果 我 无聊 时 ...', 'label': 1}

    map_kwargs = {
        'batched': True,  # 启用批处理模式，tokenize 接收的是 batch 字典
        'batch_size': 512,  # 每批 512 个样本
        'remove_columns': ['idx', 'context', 'label']  # 处理后移除这三列
    }
    REWARD_TOKEN_ID = tokenizer.eos_token_id
    print(REWARD_TOKEN_ID)
    # 在 map 时绑定 tokenizer
    tokenize_with_tokenizer = partial(tokenize, tokenizer=tokenizer,REWARD_TOKEN_ID = REWARD_TOKEN_ID)

    # 对训练集和验证集应用 map
    tokenized_dataset_train = dataset["train"].map(tokenize_with_tokenizer, **map_kwargs)
    tokenized_dataset_val = dataset["valid"].map(tokenize_with_tokenizer, **map_kwargs)

    print(tokenized_dataset_train[0])
    print(tokenized_dataset_val[0])


    # 对训练集和验证集应用过滤函数
    filtered_dataset_train = tokenized_dataset_train.filter(filter_short_samples)
    filtered_dataset_val = tokenized_dataset_val.filter(filter_short_samples)
    print(f"训练集过滤前样本数量: {len(tokenized_dataset_train)}")
    print(f"训练集过滤后样本数量: {len(filtered_dataset_train)}")
    print(f"验证集过滤前样本数量: {len(tokenized_dataset_val)}")
    print(f"验证集过滤后样本数量: {len(filtered_dataset_val)}")

    # 放入torch
    import torch
    filtered_dataset_train.set_format(type='torch')
    filtered_dataset_val.set_format(type='torch')
    print(filtered_dataset_train[0])
    print(filtered_dataset_val[0])


    return filtered_dataset_train, filtered_dataset_val, dataset["test"]

data_path = ["../SFT微调大模型/data_raw/train.jsonl","../SFT微调大模型/data_raw/valid.jsonl","../SFT微调大模型/data_raw/test.jsonl"]
tokenizer = AutoTokenizer.from_pretrained(model_path)
filtered_dataset_train, filtered_dataset_val, dataset_test = load_data(data_path=data_path,tokenizer=tokenizer)

{'idx': 0, 'context': '死囚爱刽子手女贼爱衙役我们爱你们难道还有别的选择没想到胡军除了蓝宇还有东宫西宫我个去阿兰这样真他nia恶心爱个P分明只是欲', 'label': 1}
50256
{'input_ids': [29826, 119, 32368, 248, 163, 230, 109, 26344, 121, 36310, 33699, 233, 42637, 164, 112, 120, 163, 230, 109, 26193, 247, 37605, 117, 22755, 239, 20015, 105, 163, 230, 109, 19526, 254, 20015, 105, 49694, 122, 34402, 241, 32573, 246, 17312, 231, 26344, 104, 21410, 34460, 231, 162, 233, 102, 162, 110, 94, 46349, 111, 26344, 108, 47797, 94, 37863, 249, 165, 247, 97, 12859, 228, 164, 241, 251, 22522, 229, 32573, 246, 17312, 231, 10310, 250, 22522, 104, 164, 98, 123, 22522, 104, 22755, 239, 10310, 103, 43889, 119, 165, 246, 123, 17739, 108, 32573, 247, 43718, 115, 40367, 253, 20015, 244, 18142, 162, 223, 114, 33232, 225, 163, 230, 109, 10310, 103, 47, 26344, 228, 23626, 236, 20998, 103, 42468, 162, 105, 110, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [14]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
# 将 eos token 作为 pad token
tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorWithPadding(tokenizer)
dataloader_params = {
    'batch_size': 16,
    'shuffle': True,
    'collate_fn': data_collator
}
train_dataloader = DataLoader(filtered_dataset_train, **dataloader_params)
val_dataloader = DataLoader(filtered_dataset_val, **dataloader_params)

batch = next(iter(train_dataloader))
print(batch.keys())

print(batch['input_ids'][1])
print(batch['attention_mask'][1])
print(batch['score'][1])
print(batch['score_index'][1])
print(tokenizer.decode(batch['input_ids'][1]))
print(batch['attention_mask'][1].nonzero()[-1])

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


dict_keys(['input_ids', 'attention_mask', 'score', 'score_index'])
tensor([  164, 50159, 47078, 35050,   107,   242,   164,   122,   225, 46479,
          245, 25001,   245, 12859,   110, 46349,   227, 27950,   109, 33232,
          245, 30298,   100, 19526,   228, 35707,   253,   162,   253,   241,
        27950,   249, 32573,   246, 42468, 40367,   233, 26344,   108,   164,
          229,   103, 32432,   109,   163,   230,   114, 12859,   110, 10310,
          118, 12859,   228, 17312,   118,   161,   247,   101, 21689, 20046,
          253,   164,   229,   103, 32432,   109, 17312,   222, 28938,   236,
        31660, 22755,   246, 33768,   114,   161,   222,   247, 21410,   163,
           94,   106,   165,   251,   252, 30585,   116, 35707,   253, 21689,
        18796,   113, 37605,   109,   165,   227,   235, 20046,   238, 20046,
          253, 38834,   165,   242,   247, 20998,    99, 13783,   244, 22887,
          237, 29826,    96, 13783,   103, 20015,    98, 28938,   236, 2362

In [15]:
outputs = model(batch['input_ids'], batch['attention_mask'])
print(outputs.shape)

torch.Size([16, 145])


In [16]:
#论文里面是只训练了一个epoch，奖励模型的训练通常需要较少的epoch，因为它们通常是用来评估生成模型的输出，而不是直接生成文本。过多的训练可能会导致奖励模型过拟合，从而无法有效地评估生成模型的输出。
'''
发现训练全部模型的结果并不好
'''
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
# 二分类交叉熵损失
criterion = nn.BCELoss()
num_epochs = 1


In [22]:
#评估
def validate():
    model.eval()
    total_loss = 0
    for i, batch in enumerate(val_dataloader):
        inputs = batch.to(device)
        model_inputs = {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask']
        }
        with torch.no_grad():
            # 对输出进行评分
            scores = model(**model_inputs)
            # 批次中每条数据的索引
            batch_indices = torch.arange(scores.shape[0])
            # 根据索引拿出评分，也就是reward token的评分
            score = scores[batch_indices, inputs['score_index']]
            # 目标评分，0 或者 1 。
            target = inputs['score']  #需要注意的是这里的评分都是大于0的，但是可以通过调整reward token的位置来让模型学会给负向样本打分更低，给正向样本打分更高。
            # 计算误差
            loss = criterion(score, target)
        total_loss += loss.item()
    print('validation loss:', total_loss / len(val_dataloader))

#训练
# 加载已有的模型
import os
if os.path.exists('models/reward_model.pt'):
    model.load_state_dict(torch.load('models/reward_model.pt'))
    print('reward_model.pt loaded')
model.to(device)

validate()
for epoch in range(num_epochs):
    model.train()
    for i, batch in enumerate(train_dataloader):
        inputs = batch.to(device)
        model_inputs = {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask']
        }
        scores = model(**model_inputs)
        batch_indices = torch.arange(scores.shape[0])
        score = scores[batch_indices, inputs['score_index']]
        target = inputs['score']
        loss = criterion(score, target)
        #清空梯度 ⟶ 反向传播计算梯度 ⟶ 更新参数
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(loss.item())
    validate()

reward_model.pt loaded
validation loss: 0.5047243722947314
0.2125544399023056
0.421308696269989
0.4060438275337219
0.36137261986732483
0.7156973481178284
0.3925495743751526
0.30080944299697876
0.32701575756073
0.3827349543571472
1.4283009767532349
0.47533461451530457
0.2742094099521637
1.0745210647583008
0.4963409900665283
0.540213406085968
0.6969257593154907
0.507900059223175
0.3951638340950012
1.1012358665466309
0.6594716310501099
0.9284037351608276
0.6231029629707336
0.4134991466999054
0.3676168620586395
0.47565919160842896
0.4633493423461914
0.4891616702079773
0.34889012575149536
0.4310494363307953
0.9225447177886963
0.6681280136108398
0.44762155413627625
0.47820839285850525
0.48620080947875977
0.29825711250305176
0.45042896270751953
0.6131687760353088
0.4152474105358124
0.46637606620788574
0.4403534531593323
0.4582712650299072
0.2455502450466156
0.33589038252830505
0.40457937121391296
0.5547987222671509
0.6579254269599915
0.4767494201660156
0.3290615677833557
0.40159690380096436
0

In [20]:
torch.save(model.state_dict(), 'models/reward_model.pt')

In [21]:
#评估
#加载已有的模型
# import os
# if os.path.exists('models/reward_model.pt'):
#     model.load_state_dict(torch.load('models/reward_model.pt'))
#     print('reward_model.pt loaded')
model.to(device)
validate()
print(model)

validation loss: 0.5048249583924189
GPT2RewardHead(
  (llm): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(50304, 768)
      (wpe): Embedding(1024, 768)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-11): 12 x GPT2Block(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D()
            (c_proj): Conv1D()
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D()
            (c_proj): Conv1D()
            (act): FastGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_features=768, out_features=50304, bias=False)
  